# EmpulseAI — Multi-Class Sentiment Analysis of Company Reviews
## Using Ensemble Deep Learning Models for Employer Branding
### Master's Thesis | Module 37284 — Marketing Research
### ✅ REAL DATASET VERSION — Glassdoor Job Reviews (Kaggle)

---

**Dataset Source:** [Glassdoor Job Reviews](https://www.kaggle.com/datasets/davidgauthier/glassdoor-job-reviews) by David Gauthier (Kaggle)

**Data Provenance Note:** The full Kaggle dataset contains 838,566 real employee reviews across 428 companies. For computational feasibility on Colab's free tier, a stratified random sample of 7,500 reviews (2,500 per sentiment class) was drawn from the real dataset. Sentiment labels were derived from the `overall_rating` field (1-2 stars → Negative, 3 stars → Neutral, 4-5 stars → Positive), and review text was constructed by concatenating the `pros` and `cons` fields. This sampling and labelling methodology is fully documented here for reproducibility.

**Research Objective:** Develop and evaluate a RoBERTa + Bi-LSTM ensemble model for multi-class sentiment classification (Positive / Neutral / Negative) of company reviews to support employer branding intelligence.

---

### Table of Contents
1. Environment Setup & Library Installation
2. Real Dataset Loading & EDA
3. Data Preprocessing Pipeline
4. Model 1 — RoBERTa Fine-Tuning
5. Model 2 — Bi-LSTM with GloVe Embeddings
6. Ensemble Strategy Implementation & Comparison
7. Results & Evaluation (Confusion Matrix, F1, Accuracy)
8. SHAP Interpretability Analysis
9. Web Application Integration


## Section 1 — Environment Setup & Library Installation

In [ ]:
!pip install transformers torch torchvision torchaudio --quiet
!pip install datasets scikit-learn imbalanced-learn --quiet
!pip install shap matplotlib seaborn pandas numpy --quiet

print("✅ All libraries installed successfully!")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizer, RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, precision_score, recall_score
)
from sklearn.linear_model import LogisticRegression

import re
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from imblearn.over_sampling import SMOTE
import shap

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Libraries imported successfully!")
print(f"📱 Device: {device}")
print(f"🔥 CUDA Available: {torch.cuda.is_available()}")


## Section 2 — Real Dataset Loading (Glassdoor — Kaggle)

In [ ]:
# ─── Upload the real dataset sample ─────────────────────
# This file is a stratified sample (7,500 rows) drawn from the real
# Glassdoor Job Reviews dataset (838,566 rows) available at:
# https://www.kaggle.com/datasets/davidgauthier/glassdoor-job-reviews
#
# Upload 'glassdoor_sample_for_colab.csv' using the Files panel on the left
# (folder icon → Upload), then run this cell.

from google.colab import files
print("📤 Please upload 'glassdoor_sample_for_colab.csv' when prompted...")
uploaded = files.upload()


In [ ]:
df = pd.read_csv('glassdoor_sample_for_colab.csv')

print(f"✅ Real dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\n📋 Source: Glassdoor Job Reviews (Kaggle) — stratified sample")
print(f"\nFirst 5 rows:")
df.head()


In [ ]:
print("=" * 55)
print("        REAL DATASET OVERVIEW — EDA")
print("=" * 55)
print(f"Total Reviews     : {len(df):,}")
print(f"Unique Companies  : {df['company'].nunique()}")
print(f"Missing Values    : {df.isnull().sum().sum()}")
print(f"\nSentiment Distribution:")
print(df['sentiment'].value_counts())
print(f"\nRating Distribution:")
print(df['overall_rating'].value_counts().sort_index())


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('EmpulseAI — Exploratory Data Analysis (Real Glassdoor Data)', fontsize=16, fontweight='bold')

sentiment_counts = df['sentiment'].value_counts()
colors = ['#00e5a0', '#ffd166', '#ff6b6b']
axes[0,0].pie(sentiment_counts.values, labels=sentiment_counts.index,
               colors=colors, autopct='%1.1f%%', startangle=90)
axes[0,0].set_title('Sentiment Class Distribution', fontweight='bold')

rating_counts = df['overall_rating'].value_counts().sort_index()
axes[0,1].bar(rating_counts.index, rating_counts.values,
               color=['#ff6b6b','#ff9966','#ffd166','#90ee90','#00e5a0'])
axes[0,1].set_title('Star Rating Distribution', fontweight='bold')
axes[0,1].set_xlabel('Star Rating')
axes[0,1].set_ylabel('Count')

top_companies = df['company'].value_counts().head(10).index
company_sentiment = df[df['company'].isin(top_companies)].groupby(['company','sentiment']).size().unstack(fill_value=0)
company_sentiment.plot(kind='bar', ax=axes[1,0], color=colors, rot=45)
axes[1,0].set_title('Sentiment by Company (Top 10)', fontweight='bold')
axes[1,0].set_xlabel('')
axes[1,0].legend(loc='upper right', fontsize=8)

sentiment_rating = df.groupby(['sentiment','overall_rating']).size().unstack(fill_value=0)
sentiment_rating.plot(kind='bar', ax=axes[1,1], rot=0)
axes[1,1].set_title('Rating vs Sentiment (RQ4)', fontweight='bold')
axes[1,1].set_xlabel('Sentiment')

plt.tight_layout()
plt.savefig('eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA visualizations complete (real data)!")


## Section 3 — Data Preprocessing Pipeline (Methodology 3.3)

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = clean_text(text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

print("⚙️  Running preprocessing pipeline on real reviews...")
df['clean_review'] = df['review'].apply(preprocess_text)
df['review_length'] = df['clean_review'].apply(lambda x: len(x.split()))

print(f"✅ Preprocessing complete!")
print(f"Avg review length: {df['review_length'].mean():.1f} tokens")
print(f"95th percentile  : {int(np.percentile(df['review_length'], 95))} tokens")
print(f"\nSample preprocessed review:")
print(f"  Original : {df['review'].iloc[0][:150]}")
print(f"  Cleaned  : {df['clean_review'].iloc[0][:150]}")


In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['sentiment'])
label_map = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"Label mapping: {label_map}")

X = df['clean_review'].values
y = df['label'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

print(f"\n✅ Stratified Split (70/15/15):")
print(f"  Train : {len(X_train):,} samples")
print(f"  Val   : {len(X_val):,} samples")
print(f"  Test  : {len(X_test):,} samples")

print(f"\n⚖️  Class distribution in train set:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {le.classes_[u]}: {c}")
print("\n(Sample was pre-balanced at 2,500/class during stratified sampling from the full 838K-row real dataset)")


## Section 4 — Model 1: RoBERTa Fine-Tuning (RQ2)

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

print("📥 Loading RoBERTa-base tokenizer and model...")
MODEL_NAME = 'roberta-base'
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
roberta_model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
roberta_model = roberta_model.to(device)

BATCH_SIZE = 16
MAX_LEN = 128

train_dataset = ReviewDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = ReviewDataset(X_val, y_val, tokenizer, MAX_LEN)
test_dataset  = ReviewDataset(X_test, y_test, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"✅ RoBERTa model loaded: {sum(p.numel() for p in roberta_model.parameters()):,} parameters")
print(f"✅ DataLoaders ready — Batch size: {BATCH_SIZE}, Max length: {MAX_LEN}")


In [ ]:
def train_roberta(model, train_loader, val_loader, epochs=1, lr=2e-5):
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

    train_losses, val_losses, val_f1s = [], [], []
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
        avg_train_loss = total_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        model.eval()
        val_preds, val_true = [], []
        val_loss_total = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                val_loss_total += outputs.loss.item()
                preds = torch.argmax(outputs.logits, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(labels.cpu().numpy())
        val_f1 = f1_score(val_true, val_preds, average='macro')
        val_acc = accuracy_score(val_true, val_preds)
        val_losses.append(val_loss_total/len(val_loader))
        val_f1s.append(val_f1)
        print(f"  Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {val_loss_total/len(val_loader):.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")
    return train_losses, val_losses, val_f1s

print("🚀 Starting RoBERTa fine-tuning on REAL data...")
print("   Strategy: Staged fine-tuning (frozen → full)")
print("   NOTE: epochs=1 used for Colab free-tier CPU/GPU time limits.")
print("   Increase epochs for production-quality results if compute allows.")
print("-" * 55)

for param in roberta_model.roberta.parameters():
    param.requires_grad = False
print("📌 Stage 1: Training classifier head only...")
train_losses_s1, val_losses_s1, val_f1s_s1 = train_roberta(roberta_model, train_loader, val_loader, epochs=1)

for param in roberta_model.roberta.parameters():
    param.requires_grad = True
print("\n📌 Stage 2: Full model fine-tuning...")
train_losses_s2, val_losses_s2, val_f1s_s2 = train_roberta(roberta_model, train_loader, val_loader, epochs=2, lr=2e-5)

print("\n✅ RoBERTa fine-tuning complete!")


In [ ]:
def evaluate_model(model, loader, model_name="Model"):
    model.eval()
    all_preds, all_probs, all_true = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1)
            preds = torch.argmax(probs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_true.extend(labels.cpu().numpy())
    f1 = f1_score(all_true, all_preds, average='macro')
    acc = accuracy_score(all_true, all_preds)
    prec = precision_score(all_true, all_preds, average='macro')
    rec = recall_score(all_true, all_preds, average='macro')
    print(f"\n{'='*55}\n  {model_name} — Test Results (REAL DATA)\n{'='*55}")
    print(f"  Macro F1-Score : {f1:.4f}")
    print(f"  Accuracy       : {acc:.4f}")
    print(f"  Precision      : {prec:.4f}")
    print(f"  Recall         : {rec:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(all_true, all_preds, target_names=le.classes_))
    return all_preds, np.array(all_probs), all_true

roberta_preds, roberta_probs, y_true = evaluate_model(roberta_model, test_loader, "RoBERTa-base (Fine-tuned, REAL DATA)")


## Section 5 — Model 2: Bi-LSTM with GloVe Embeddings

In [ ]:
import os, urllib.request, zipfile

GLOVE_FILE = "glove.6B.300d.txt"
if not os.path.exists(GLOVE_FILE):
    print("📥 Downloading GloVe 300d embeddings...")
    urllib.request.urlretrieve("http://nlp.stanford.edu/data/glove.6B.zip", "glove.6B.zip")
    with zipfile.ZipFile("glove.6B.zip", 'r') as z:
        z.extract(GLOVE_FILE)
    print("✅ GloVe downloaded!")
else:
    print("✅ GloVe already available!")

from tensorflow.keras.preprocessing.text import Tokenizer as KerasTokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_VOCAB = 20000
MAX_SEQ = 100
EMBED_DIM = 300

keras_tokenizer = KerasTokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
keras_tokenizer.fit_on_texts(X_train)

def texts_to_sequences_padded(texts):
    seqs = keras_tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_SEQ, padding='post', truncating='post')

X_train_seq = texts_to_sequences_padded(X_train)
X_val_seq   = texts_to_sequences_padded(X_val)
X_test_seq  = texts_to_sequences_padded(X_test)

print("📥 Loading GloVe vectors into embedding matrix...")
glove_embeddings = {}
with open(GLOVE_FILE, encoding='utf-8') as f:
    for line in f:
        values = line.split()
        glove_embeddings[values[0]] = np.asarray(values[1:], dtype='float32')

word_index = keras_tokenizer.word_index
vocab_size = min(len(word_index)+1, MAX_VOCAB)
embed_matrix = np.zeros((vocab_size, EMBED_DIM))
covered, total = 0, 0
for word, idx in word_index.items():
    if idx < MAX_VOCAB:
        total += 1
        vec = glove_embeddings.get(word)
        if vec is not None:
            embed_matrix[idx] = vec
            covered += 1

print(f"✅ Embedding matrix: {embed_matrix.shape}")
print(f"   GloVe coverage : {covered}/{total} = {covered/total*100:.1f}%")


In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, embed_matrix, hidden_dim=256, num_layers=2, num_classes=3, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight = nn.Parameter(torch.FloatTensor(embed_matrix), requires_grad=True)
        self.bilstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                               bidirectional=True, batch_first=True, dropout=dropout)
        self.spatial_dropout = nn.Dropout2d(dropout)
        self.fc1 = nn.Linear(hidden_dim*2, 256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(256, num_classes)
    def forward(self, x):
        embedded = self.embedding(x)
        embedded = self.spatial_dropout(embedded.unsqueeze(2)).squeeze(2)
        lstm_out, _ = self.bilstm(embedded)
        pooled = torch.max(lstm_out, dim=1)[0]
        out = self.fc1(pooled); out = self.relu(out); out = self.dropout(out); out = self.fc2(out)
        return out

bilstm_model = BiLSTMClassifier(vocab_size, EMBED_DIM, embed_matrix, 256, 2, 3, 0.3).to(device)
total_params = sum(p.numel() for p in bilstm_model.parameters())
print(f"✅ Bi-LSTM model initialized! Total parameters: {total_params:,}")


In [ ]:
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.LongTensor(X); self.y = torch.LongTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_seq_loader = DataLoader(SeqDataset(X_train_seq, y_train), batch_size=64, shuffle=True)
val_seq_loader   = DataLoader(SeqDataset(X_val_seq, y_val), batch_size=64)
test_seq_loader  = DataLoader(SeqDataset(X_test_seq, y_test), batch_size=64)

class_counts = np.bincount(y_train)
class_weights = torch.FloatTensor(1.0/class_counts).to(device)
class_weights = class_weights / class_weights.sum()
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(bilstm_model.parameters(), lr=1e-3)

print("🚀 Training Bi-LSTM model on REAL data (8 epochs)...")
print("-"*55)
bilstm_train_losses, bilstm_val_f1s = [], []
for epoch in range(8):
    bilstm_model.train()
    total_loss = 0
    for X_batch, y_batch in train_seq_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = bilstm_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bilstm_model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    bilstm_model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_seq_loader:
            X_batch = X_batch.to(device)
            outputs = bilstm_model(X_batch)
            preds = torch.argmax(outputs, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_true.extend(y_batch.numpy())
    val_f1 = f1_score(val_true, val_preds, average='macro')
    avg_loss = total_loss/len(train_seq_loader)
    bilstm_train_losses.append(avg_loss); bilstm_val_f1s.append(val_f1)
    print(f"  Epoch {epoch+1:2d}/8 | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}")

bilstm_model.eval()
bilstm_preds_list, bilstm_probs_list = [], []
with torch.no_grad():
    for X_batch, _ in test_seq_loader:
        X_batch = X_batch.to(device)
        outputs = bilstm_model(X_batch)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)
        bilstm_preds_list.extend(preds.cpu().numpy())
        bilstm_probs_list.extend(probs.cpu().numpy())

bilstm_preds = np.array(bilstm_preds_list)
bilstm_probs = np.array(bilstm_probs_list)
bilstm_f1 = f1_score(y_true, bilstm_preds, average='macro')
bilstm_acc = accuracy_score(y_true, bilstm_preds)
print(f"\n✅ Bi-LSTM Test (REAL DATA) | F1: {bilstm_f1:.4f} | Acc: {bilstm_acc:.4f}")


## Section 6 — Ensemble Strategy Implementation & Comparison (RQ3)

In [ ]:
print("🔗 Implementing 4 Ensemble Strategies on REAL DATA results...")
results = {}

hard_preds = []
for r, b in zip(roberta_preds, bilstm_preds):
    votes = [r, b]
    hard_preds.append(max(set(votes), key=votes.count))
hard_preds = np.array(hard_preds)
results['Hard Voting'] = {'preds': hard_preds, 'f1': f1_score(y_true, hard_preds, average='macro'),
    'acc': accuracy_score(y_true, hard_preds), 'prec': precision_score(y_true, hard_preds, average='macro'),
    'rec': recall_score(y_true, hard_preds, average='macro')}

soft_probs = (roberta_probs + bilstm_probs) / 2
soft_preds = np.argmax(soft_probs, axis=1)
results['Soft Voting'] = {'preds': soft_preds, 'f1': f1_score(y_true, soft_preds, average='macro'),
    'acc': accuracy_score(y_true, soft_preds), 'prec': precision_score(y_true, soft_preds, average='macro'),
    'rec': recall_score(y_true, soft_preds, average='macro')}

w1, w2 = 0.6, 0.4
weighted_probs = (w1*roberta_probs) + (w2*bilstm_probs)
weighted_preds = np.argmax(weighted_probs, axis=1)
results['Weighted Avg.'] = {'preds': weighted_preds, 'f1': f1_score(y_true, weighted_preds, average='macro'),
    'acc': accuracy_score(y_true, weighted_preds), 'prec': precision_score(y_true, weighted_preds, average='macro'),
    'rec': recall_score(y_true, weighted_preds, average='macro')}

meta_features = np.hstack([roberta_probs, bilstm_probs])
split_idx = int(0.8*len(meta_features))
meta_clf = LogisticRegression(max_iter=1000, random_state=SEED)
meta_clf.fit(meta_features[:split_idx], np.array(y_true)[:split_idx])
stack_preds = meta_clf.predict(meta_features)
results['Stacking (LR)'] = {'preds': stack_preds, 'f1': f1_score(y_true, stack_preds, average='macro'),
    'acc': accuracy_score(y_true, stack_preds), 'prec': precision_score(y_true, stack_preds, average='macro'),
    'rec': recall_score(y_true, stack_preds, average='macro')}

results['RoBERTa alone'] = {'preds': roberta_preds, 'f1': f1_score(y_true, roberta_preds, average='macro'),
    'acc': accuracy_score(y_true, roberta_preds), 'prec': precision_score(y_true, roberta_preds, average='macro'),
    'rec': recall_score(y_true, roberta_preds, average='macro')}
results['Bi-LSTM alone'] = {'preds': bilstm_preds, 'f1': f1_score(y_true, bilstm_preds, average='macro'),
    'acc': accuracy_score(y_true, bilstm_preds), 'prec': precision_score(y_true, bilstm_preds, average='macro'),
    'rec': recall_score(y_true, bilstm_preds, average='macro')}

best_strategy = max(results.items(), key=lambda x: x[1]['f1'])[0]
print("\n" + "="*65)
print(f"{'Strategy':<20} {'Macro F1':>10} {'Accuracy':>10} {'Precision':>10} {'Recall':>10}")
print("="*65)
for name, m in results.items():
    marker = " ← BEST" if name == best_strategy else ""
    print(f"{name:<20} {m['f1']:>10.4f} {m['acc']:>10.4f} {m['prec']:>10.4f} {m['rec']:>10.4f}{marker}")
print("="*65)
print(f"\n✅ Best strategy on REAL data: {best_strategy}")


## Section 7 — Results & Evaluation

In [ ]:
best_key = max(results.items(), key=lambda x: x[1]['f1'])[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrices — Best Ensemble vs RoBERTa Alone (REAL DATA)', fontsize=14, fontweight='bold')

for ax, (name, key) in zip(axes, [(f'{best_key} (Best Ensemble)', best_key), ('RoBERTa alone (Baseline)', 'RoBERTa alone')]):
    cm = confusion_matrix(y_true, results[key]['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Negative','Neutral','Positive'],
                yticklabels=['Negative','Neutral','Positive'])
    ax.set_title(f"{name}\nMacro F1: {results[key]['f1']:.4f}", fontweight='bold')
    ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Ensemble Strategy Comparison on REAL DATA (RQ3)', fontsize=14, fontweight='bold')
names = list(results.keys())
f1s = [results[n]['f1'] for n in names]
accs = [results[n]['acc'] for n in names]
colors = ['#00e5a0' if n == best_key else '#3d7fff' if 'alone' not in n else '#ff6b6b' for n in names]

axes[0].barh(names, f1s, color=colors)
axes[0].set_title('Macro F1-Score Comparison', fontweight='bold'); axes[0].set_xlabel('Macro F1-Score')
for i, v in enumerate(f1s): axes[0].text(v+0.005, i, f'{v:.4f}', va='center', fontsize=9)

axes[1].barh(names, accs, color=colors)
axes[1].set_title('Accuracy Comparison', fontweight='bold'); axes[1].set_xlabel('Accuracy')
for i, v in enumerate(accs): axes[1].text(v+0.005, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('ensemble_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Results visualizations complete (REAL DATA)!")


In [ ]:
best_f1 = results[best_key]['f1']
ablation = {
    f'Full Ensemble ({best_key})': best_f1,
    'RoBERTa alone': results['RoBERTa alone']['f1'],
    'Bi-LSTM alone': results['Bi-LSTM alone']['f1'],
}
print("\n" + "="*60)
print("     ABLATION STUDY RESULTS (REAL DATA)")
print("="*60)
print(f"{'Configuration':<40} {'F1':>8} {'Δ vs Best':>10}")
print("-"*60)
for config, f1 in ablation.items():
    delta = f1 - best_f1
    delta_str = f'{delta:+.4f}' if abs(delta) > 0.0001 else '—'
    print(f"{config:<40} {f1:>8.4f} {delta_str:>10}")
print("="*60)
print("\nNote: with epochs reduced for Colab free-tier time limits,")
print("differences between configurations may be modest. Increasing")
print("epochs and using full GPU runtime would sharpen these gaps.")


## Section 8 — SHAP Interpretability Analysis (Objective 5)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression as LR

shap_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000, ngram_range=(1,2))),
    ('clf', LR(max_iter=1000, random_state=SEED))
])
shap_pipeline.fit(X_train, y_train)

print("🔬 Computing SHAP values for interpretability (REAL DATA)...")
explainer = shap.Explainer(shap_pipeline['clf'], shap_pipeline['tfidf'].transform(X_train[:100]))
X_test_tfidf = shap_pipeline['tfidf'].transform(X_test[:50])
shap_values = explainer(X_test_tfidf)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values[:,:,0], X_test_tfidf,
                  feature_names=shap_pipeline['tfidf'].get_feature_names_out(),
                  plot_type="bar", show=False, max_display=15)
plt.title('SHAP Feature Importance — Real Glassdoor Reviews (Positive Class)', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ SHAP analysis complete (REAL DATA)!")


In [ ]:
sample_idx = 0
sample_review_raw = df['review'].iloc[sample_idx]
sample_clean = preprocess_text(sample_review_raw)
print(f"\n🔍 Sample Real Review: '{sample_review_raw[:200]}'")

sample_tfidf = shap_pipeline['tfidf'].transform([sample_clean])
sample_shap = explainer(sample_tfidf)
pred_class = shap_pipeline.predict([sample_clean])[0]
pred_proba = shap_pipeline.predict_proba([sample_clean])[0]

print(f"\n  Predicted Sentiment : {le.classes_[pred_class] if pred_class < len(le.classes_) else pred_class}")
print(f"  Confidence Scores:")
for cls, prob in zip(shap_pipeline.classes_, pred_proba):
    label = le.classes_[cls] if cls < len(le.classes_) else cls
    bar = '█' * int(prob*30)
    print(f"    {label:<10}: {prob:.4f} {bar}")

feature_names = shap_pipeline['tfidf'].get_feature_names_out()
shap_vals = sample_shap.values[0,:,0]
top_idx = np.argsort(np.abs(shap_vals))[::-1][:10]
print(f"\n  Top SHAP Features:")
print(f"  {'Word':<20} {'SHAP Value':>12} {'Direction':>12}")
print(f"  {'-'*45}")
for idx in top_idx:
    direction = '▲ Positive' if shap_vals[idx] > 0 else '▼ Negative'
    print(f"  {feature_names[idx]:<20} {shap_vals[idx]:>12.4f} {direction:>12}")


## Section 9 — Web Application Integration

In [ ]:
import joblib

print("💾 Saving trained models (from REAL data)...")
roberta_model.save_pretrained('./empulseai_roberta')
tokenizer.save_pretrained('./empulseai_roberta')
torch.save(bilstm_model.state_dict(), 'bilstm_model.pt')
joblib.dump(keras_tokenizer, 'keras_tokenizer.pkl')
joblib.dump(le, 'label_encoder.pkl')
joblib.dump(meta_clf, 'stacking_meta_clf.pkl')

print("✅ Models saved successfully!")
print("   ./empulseai_roberta/  — RoBERTa model (trained on real Glassdoor data)")
print("   bilstm_model.pt       — Bi-LSTM weights")
print("   keras_tokenizer.pkl   — Keras tokenizer")
print("   label_encoder.pkl     — Label encoder")
print("   stacking_meta_clf.pkl — Meta classifier")


In [ ]:
print("\n" + "="*65)
print("   EMPULSEAI — FINAL RESEARCH SUMMARY (REAL DATA)")
print("="*65)
print(f"\n  Dataset   : Glassdoor Job Reviews (Kaggle, davidgauthier)")
print(f"  Full size : 838,566 real reviews / 428 companies")
print(f"  Sample    : 7,500 stratified reviews used for this run")
print(f"  Labels    : Derived from overall_rating (1-2=Neg,3=Neu,4-5=Pos)")
print(f"\n  {'Model':<25} {'Macro F1':>10} {'Accuracy':>10}")
print(f"  {'-'*48}")
for name, m in results.items():
    marker = " ✅ BEST" if name == best_key else ""
    print(f"  {name:<25} {m['f1']:>10.4f} {m['acc']:>10.4f}{marker}")
print(f"\n  Best strategy: {best_key}")
print("="*65)
